# Mental Health Atlas: Exploratory Data Analysis & Statistical Modeling

**A Comprehensive Study of Global Mental Health Trends (1990–2019)**

This Jupyter Notebook contains the complete exploratory data analysis (EDA), data cleaning, statistical modeling, and data visualization pipeline for the **Mental Health Atlas** project.

### Project Architecture & Coexistence
- **Jupyter Notebook (`mental_health_analysis.ipynb`)**: The *Data Science & Research Lab* — step-by-step data inspection, sanitization, Pandas grouping, and inline Matplotlib plotting.
- **Web Application (`app.py`)**: The *Production Presentation Layer* — an interactive web application built with Python's built-in `http.server` that serves these insights to end users without external web frameworks.

---
### Table of Contents
1. **Environment Setup & Imports**
2. **Data Ingestion & Preliminary Inspection**
3. **Data Cleaning, Standardization & Country Filtering**
4. **Longitudinal Analysis: Global Mental Health Trends (1990–2019)**
5. **Comparative Country Analysis (Prevalence & Disease Burden DALYs)**
6. **Disorder Rankings & Top-Affected Countries**
7. **The Treatment Gap & Depressive Symptom Breakdown**
8. **Data Coverage, Scarcity & Epidemiological Limitations**
9. **Coexistence Architecture & Launching the Web Server**

## 1. Environment Setup & Imports

We import `pandas` for data manipulation, `numpy` for mathematical calculations, and `matplotlib.pyplot` for data visualization. We configure inline plotting and define a curated color palette matching the editorial theme of the web atlas.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configure inline plotting and high resolution
%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans', 'Arial']

# Editorial color palette (matching the web atlas)
GREEN = "#2f5d46"
SAGE = "#a9b8a0"
ACCENT = "#b5683a"
CHARCOAL = "#2b2d2a"
LINE_COLORS = [GREEN, ACCENT, "#7d9a78", CHARCOAL, "#d4a373"]

CONDITIONS = [
    "Depressive disorders", 
    "Anxiety disorders", 
    "Bipolar disorder", 
    "Schizophrenia", 
    "Eating disorders"
]

DATA_FOLDER = os.path.join(".", "data")
print("Environment initialized successfully.")
print("Target Conditions:", CONDITIONS)

## 2. Data Ingestion & Preliminary Inspection

We load all 7 CSV files originating from the Institute for Health Metrics and Evaluation (IHME) Global Burden of Disease (GBD) study (published via Our World in Data / Kaggle).

In [ ]:
DATASETS_META = [
    ("prevalence", "1_prevalence.csv", "Prevalence of mental illnesses"),
    ("burden", "2_burden_dalys.csv", "Disease burden (DALYs)"),
    ("coverage_dep", "3_coverage_depression.csv", "Data coverage: major depression"),
    ("coverage_all", "4_coverage_all.csv", "Data coverage: all disorders"),
    ("treatment", "5_treatment_gap.csv", "Anxiety treatment gap"),
    ("symptoms", "6_symptoms_us.csv", "Depressive symptoms in the US"),
    ("countries_data", "7_countries_with_data.csv", "Countries with primary data"),
]

data = {}
for key, file_name, title in DATASETS_META:
    file_path = os.path.join(DATA_FOLDER, file_name)
    data[key] = pd.read_csv(file_path)
    print(f"[Loaded] {key:<15} ({file_name}): {data[key].shape[0]:>5} rows, {data[key].shape[1]:>2} columns")

In [ ]:
# Inspect sample records from 1_prevalence.csv
print("Sample of Prevalence Data (First 5 Rows):")
data["prevalence"].head()

## 3. Data Cleaning, Standardization & Country Filtering

Three essential data cleaning steps are required:
1. **Column Renaming**: The raw IHME column headers are verbose strings (e.g. `"Depressive disorders (share of population) - Sex: Both - Age: Age-standardized"`). We rename them to clean, standard names.
2. **Type Coercion**: In `4_coverage_all.csv`, the Schizophrenia column contains non-numeric characters. We use `pd.to_numeric(..., errors='coerce')` to parse them into floats, converting invalid values to `NaN`.
3. **Country Filtering (`only_countries`)**: The dataset mixes real countries with regional entities (e.g. `"World"`, `"South Asia"`). We filter for rows where `Code` is present and does not start with `OWID`.

In [ ]:
# 1. Shorten verbose column names in prevalence and burden datasets
data["prevalence"] = data["prevalence"].rename(columns={
    data["prevalence"].columns[3]: "Schizophrenia",
    data["prevalence"].columns[4]: "Depressive disorders",
    data["prevalence"].columns[5]: "Anxiety disorders",
    data["prevalence"].columns[6]: "Bipolar disorder",
    data["prevalence"].columns[7]: "Eating disorders"
})

data["burden"] = data["burden"].rename(columns={
    data["burden"].columns[3]: "Depressive disorders",
    data["burden"].columns[4]: "Schizophrenia",
    data["burden"].columns[5]: "Bipolar disorder",
    data["burden"].columns[6]: "Eating disorders",
    data["burden"].columns[7]: "Anxiety disorders"
})

# 2. Convert Schizophrenia in 4_coverage_all to numeric floats
data["coverage_all"]["Schizophrenia"] = pd.to_numeric(
    data["coverage_all"]["Schizophrenia"], errors="coerce"
)

# 3. Country filtering function
def only_countries(table):
    """Retain rows corresponding to genuine countries (valid ISO codes, non-OWID aggregates)."""
    rows = table[table["Code"].notna()]
    rows = rows[~rows["Code"].astype(str).str.startswith("OWID")]
    return rows

country_prev = only_countries(data["prevalence"])
country_burden = only_countries(data["burden"])
country_names = sorted(country_prev["Entity"].unique())

print(f"Total unique genuine countries: {len(country_names)}")
print(f"Country prevalence records: {len(country_prev)}")
print(f"Country burden records: {len(country_burden)}")

## 4. Longitudinal Analysis: Global Mental Health Trends (1990–2019)

We calculate the cross-country annual average prevalence for each of the five disorders from 1990 to 2019.

In [ ]:
# Calculate cross-country yearly average for each condition
yearly_avg = country_prev.groupby("Year")[CONDITIONS].mean()
print("Yearly Cross-Country Averages (1990–2019):")
yearly_avg.head(10)

In [ ]:
# Plot: Global trends over time (1990–2019)
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor("#faf6ee")
ax.set_facecolor("#faf6ee")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="#e1dccf", linewidth=0.8)
ax.set_axisbelow(True)

for i, condition in enumerate(CONDITIONS):
    ax.plot(yearly_avg.index, yearly_avg[condition], label=condition,
            color=LINE_COLORS[i], linewidth=2.5)

ax.set_title("Average Prevalence Across Countries (1990–2019)", fontsize=14, color=GREEN, family="serif", pad=15)
ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("% of Population", fontsize=11)
ax.legend(frameon=False, fontsize=10)
plt.tight_layout()
plt.show()

### 30-Year Relative Percentage Change
Next, we compute how much each condition grew or declined relative to its 1990 baseline:
$$\text{Percentage Change} = \frac{\text{Value}_{2019} - \text{Value}_{1990}}{\text{Value}_{1990}} \times 100$$

In [ ]:
first_year = yearly_avg.iloc[0]
last_year = yearly_avg.iloc[-1]
pct_change = ((last_year - first_year) / first_year) * 100

fig, ax = plt.subplots(figsize=(9, 4.5))
fig.patch.set_facecolor("#faf6ee")
ax.set_facecolor("#faf6ee")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="#e1dccf", linewidth=0.8)
ax.set_axisbelow(True)

bar_colors = [ACCENT if val >= 0 else GREEN for val in pct_change]
bars = ax.bar(CONDITIONS, pct_change, color=bar_colors, width=0.55)
ax.axhline(0, color=CHARCOAL, linewidth=0.8)

for bar in bars:
    h = bar.get_height()
    va = 'bottom' if h >= 0 else 'top'
    ax.annotate(f"{h:+.2f}%",
                xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 4 if h >= 0 else -12),
                textcoords="offset points",
                ha='center', va=va, fontsize=10, fontweight='bold')

ax.set_title("Percentage Change in Cross-Country Average (1990 to 2019)", fontsize=14, color=GREEN, family="serif", pad=15)
ax.set_ylabel("% Change", fontsize=11)
plt.xticks(rotation=15, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Decade snapshot table (1990, 2000, 2010, 2019)
snapshot = yearly_avg.loc[[1990, 2000, 2010, 2019]].round(3)
print("Decade Snapshot of Mean Prevalence (% of Population):")
snapshot

## 5. Comparative Country Analysis

We compare historical trajectories and 2019 disease burden metrics across three major nations: **India**, the **United States**, and **Japan**.

In [ ]:
def plot_country_comparison(countries, condition):
    """Plot comparative prevalence trajectory for selected countries."""
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor("#faf6ee")
    ax.set_facecolor("#faf6ee")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="#e1dccf", linewidth=0.8)
    ax.set_axisbelow(True)

    for i, country in enumerate(countries):
        cdata = country_prev[country_prev["Entity"] == country]
        ax.plot(cdata["Year"], cdata[condition], label=country,
                color=LINE_COLORS[i % len(LINE_COLORS)], linewidth=2.5)

    ax.set_title(f"{condition}: Country Comparison (1990–2019)", fontsize=14, color=GREEN, family="serif", pad=15)
    ax.set_xlabel("Year", fontsize=11)
    ax.set_ylabel("% of Population", fontsize=11)
    ax.legend(frameon=False, fontsize=10)
    plt.tight_layout()
    plt.show()

plot_country_comparison(["India", "United States", "Japan"], "Depressive disorders")
plot_country_comparison(["India", "United States", "Japan"], "Anxiety disorders")

In [ ]:
# Grouped Bar Chart: 2019 Disease Burden (DALYs per 100,000)
def plot_country_burden(countries):
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor("#faf6ee")
    ax.set_facecolor("#faf6ee")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="#e1dccf", linewidth=0.8)
    ax.set_axisbelow(True)

    positions = np.arange(len(CONDITIONS))
    bar_width = 0.8 / len(countries)

    for i, country in enumerate(countries):
        rows = country_burden[(country_burden["Entity"] == country) & (country_burden["Year"] == 2019)]
        if not rows.empty:
            vals = rows[CONDITIONS].values[0]
            ax.bar(positions + i * bar_width, vals, bar_width, label=country, color=LINE_COLORS[i % len(LINE_COLORS)])

    ax.set_xticks(positions + bar_width * (len(countries) - 1) / 2)
    ax.set_xticklabels(CONDITIONS, fontsize=9)
    ax.set_ylabel("DALYs per 100,000", fontsize=11)
    ax.set_title("Disease Burden by Disorder in 2019 (DALYs per 100,000)", fontsize=14, color=GREEN, family="serif", pad=15)
    ax.legend(frameon=False, fontsize=10)
    plt.tight_layout()
    plt.show()

plot_country_burden(["India", "United States", "Japan"])

## 6. Disorder Rankings & Top-Affected Countries

We evaluate how conditions rank in terms of **prevalence** vs. **disease burden (DALYs)**. 

> **Key Clinical Insight:** Although Anxiety disorders have a slightly higher global prevalence than Depressive disorders, Depression causes nearly **double the DALY loss**, because its disability weight and impact on functional capacity are substantially higher.

In [ ]:
latest_p = country_prev[country_prev["Year"] == 2019]
latest_b = country_burden[country_burden["Year"] == 2019]

avg_prev = latest_p[CONDITIONS].mean().sort_values()
avg_burd = latest_b[CONDITIONS].mean().sort_values()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for ax in (ax1, ax2):
    ax.set_facecolor("#faf6ee")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="x", color="#e1dccf", linewidth=0.8)
    ax.grid(axis="y", visible=False)
fig.patch.set_facecolor("#faf6ee")

ax1.barh(avg_prev.index, avg_prev.values, color=GREEN)
ax1.set_title("Average Global Prevalence (2019)", fontsize=12, color=GREEN, family="serif")
ax1.set_xlabel("% of Population", fontsize=10)

ax2.barh(avg_burd.index, avg_burd.values, color=ACCENT)
ax2.set_title("Average Disease Burden (2019)", fontsize=12, color=GREEN, family="serif")
ax2.set_xlabel("DALYs per 100,000 People", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 countries with highest prevalence in 2019
def show_top_countries(condition, top_n=10):
    latest = country_prev[country_prev["Year"] == 2019]
    top = latest.sort_values(condition, ascending=False).head(top_n).iloc[::-1]

    fig, ax = plt.subplots(figsize=(9, 4.8))
    fig.patch.set_facecolor("#faf6ee")
    ax.set_facecolor("#faf6ee")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="x", color="#e1dccf", linewidth=0.8)
    ax.grid(axis="y", visible=False)

    ax.barh(top["Entity"], top[condition], color=ACCENT, height=0.6)
    ax.set_title(f"Top {top_n} Countries by {condition} Prevalence (2019)", fontsize=13, color=GREEN, family="serif", pad=15)
    ax.set_xlabel("% of Population", fontsize=10)
    plt.tight_layout()
    plt.show()

show_top_countries("Depressive disorders")
show_top_countries("Anxiety disorders")

## 7. The Treatment Gap & Depressive Symptom Breakdown

We analyze dataset 5 (`5_treatment_gap.csv`) and dataset 6 (`6_symptoms_us.csv`) using horizontal stacked bar charts.

In [ ]:
# Anxiety treatment gap across regions
treatment_df = data["treatment"].sort_values("Untreated, conditional")
cols = ["Potentially adequate treatment, conditional", "Other treatments, conditional", "Untreated, conditional"]
palette = [GREEN, SAGE, ACCENT]

fig, ax = plt.subplots(figsize=(10, 0.45 * len(treatment_df) + 1.8))
fig.patch.set_facecolor("#faf6ee")
ax.set_facecolor("#faf6ee")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", color="#e1dccf", linewidth=0.8)
ax.grid(axis="y", visible=False)

left = np.zeros(len(treatment_df))
for i, col in enumerate(cols):
    vals = treatment_df[col].values
    clean_label = col.replace(", conditional", "")
    ax.barh(treatment_df["Entity"], vals, left=left, color=palette[i], label=clean_label)
    left += vals

ax.set_title("Anxiety Disorder Treatment Gap Across Surveyed Regions", fontsize=13, color=GREEN, family="serif", pad=15)
ax.set_xlabel("% of People with Anxiety", fontsize=10)
ax.legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=3)
plt.tight_layout()
plt.show()

print(f"Mean Untreated Rate across Surveys: {treatment_df['Untreated, conditional'].mean():.2f}%")

In [ ]:
# Depressive symptoms reported in the United States (2014 survey)
symptoms_df = data["symptoms"].sort_values("Not at all", ascending=False)
symptom_cols = ["Nearly every day", "More than half the days", "Several days", "Not at all"]
symptom_colors = [ACCENT, "#d4a373", SAGE, "#e6eadc"]

fig, ax = plt.subplots(figsize=(10, 0.45 * len(symptoms_df) + 1.8))
fig.patch.set_facecolor("#faf6ee")
ax.set_facecolor("#faf6ee")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", color="#e1dccf", linewidth=0.8)
ax.grid(axis="y", visible=False)

left = np.zeros(len(symptoms_df))
for i, col in enumerate(symptom_cols):
    vals = symptoms_df[col].values
    ax.barh(symptoms_df["Entity"], vals, left=left, color=symptom_colors[i], label=col)
    left += vals

ax.set_title("Frequency of Reported Depressive Symptoms in the US (2014)", fontsize=13, color=GREEN, family="serif", pad=15)
ax.set_xlabel("% of Adults", fontsize=10)
ax.legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=4)
plt.tight_layout()
plt.show()

## 8. Data Coverage, Scarcity & Epidemiological Limitations

An essential requirement of scientific data analysis is understanding data limitations. Many reported statistics are model estimates rather than clinical measurements.

In [ ]:
countries_data = data["countries_data"].sort_values(
    "Number of countries with primary data on prevalence of mental disorders"
)

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor("#faf6ee")
ax.set_facecolor("#faf6ee")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", color="#e1dccf", linewidth=0.8)
ax.grid(axis="y", visible=False)

ax.barh(countries_data["Entity"], countries_data.iloc[:, 3], color=SAGE)
ax.set_title("Countries with Primary Empirical Data (2019)", fontsize=13, color=GREEN, family="serif", pad=15)
ax.set_xlabel("Number of Countries with Empirical Data", fontsize=10)
plt.tight_layout()
plt.show()

# World regions with 0% empirical data for depression in 2008
zero_regions = data["coverage_dep"][data["coverage_dep"]["Major depression"] == 0]["Entity"].tolist()
print("Regions with 0% primary empirical data coverage for depression (2008):")
for r in zero_regions:
    print(f"  - {r}")

## 9. Coexistence Architecture & Launching the Web Server

### How the Notebook and Web Application Work Together
1. **Single Source of Truth**: Both this notebook and `app.py` read directly from the shared `data/` directory.
2. **Dual-Tier Workflow**:
   - **Notebook (`mental_health_analysis.ipynb`)**: For exploratory investigation, statistical testing, and reproducible research.
   - **Application (`app.py`)**: For interactive deployment, client presentations, filtering, and cross-country comparisons via a browser GUI.

To run the web application standalone, open your terminal and execute:
```bash
python app.py
```
Then visit `http://localhost:8000` in your web browser.

In [ ]:
# Optional: Launch the web server directly inside this notebook in a background daemon thread
import threading
import time
from app import main as run_web_server

def start_server_background():
    server_thread = threading.Thread(target=run_web_server, daemon=True)
    server_thread.start()
    time.sleep(1)
    print("Mental Health Atlas web server is now active at: http://localhost:8000")

# Uncomment the line below to run the server from within Jupyter:
# start_server_background()